# PM10 Spatial Prediction over Germany 

This notebook:

1. Calculates station-level mean PM10.
2. Fits the spatial trend used in the paper.
3. Computes the empirical variogram of spatial residuals.
4. Fits a Matérn variogram 
5. Performs spatial prediction over Germany.
6. Produces the prediction mean and prediction uncertainty maps.


In [ ]:
# ============================================================
# Cell 1: Install and import packages
# ============================================================

!pip install -q openpyxl geopandas pyogrio shapely

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from pathlib import Path
from scipy.spatial.distance import pdist, squareform, cdist
from scipy.special import kv, gamma
from scipy.optimize import least_squares
from scipy.linalg import cho_factor, cho_solve

print("Packages loaded successfully.")

In [ ]:
!find /kaggle/input -maxdepth 2

In [ ]:
!find /kaggle/input/datasets/omidkarimiarman -maxdepth 3

In [ ]:
# ============================================================
# Cell 2: Paths and main settings
# ============================================================

from pathlib import Path


DATA_DIR = Path(
    "/kaggle/input/datasets/omidkarimiarman/"
    "data-pm10-germany-paper-jss-1405"
)

# 
DATA_FILE = DATA_DIR / "air_pm10_imputed_daily_2000_onwards.xlsx"

# 
GERMANY_FILE = DATA_DIR / "DE.gpkg"

# 
STATIONS_FILE = DATA_DIR / "stations.gpkg"


NUTS_FILE = DATA_DIR / "DE_NUTS1.gpkg"


OUTPUT_DIR = Path("/kaggle/working/Prediction_Maps")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


START_DATE = "2000-01-01"
END_DATE   = "2009-12-31"


GRID_NX = 250
GRID_NY = 250


SIGMA_FIXED = 1.5
SILL_FIXED = SIGMA_FIXED**2

# 
N_LAG_BINS = 10
MAX_DISTANCE_FRACTION = 0.65

# 
required_files = {
    "PM10 data": DATA_FILE,
    "Germany boundary": GERMANY_FILE,
    "Stations": STATIONS_FILE,
    "NUTS1 boundary": NUTS_FILE
}

for file_name, file_path in required_files.items():
    if not file_path.exists():
        raise FileNotFoundError(
            f"{file_name} was not found:\n{file_path}"
        )

print("All input files were found successfully.")
print("----------------------------------------")
print("Dataset directory :", DATA_DIR)
print("PM10 file         :", DATA_FILE)
print("Germany boundary  :", GERMANY_FILE)
print("Stations file     :", STATIONS_FILE)
print("NUTS1 file        :", NUTS_FILE)
print("Output directory  :", OUTPUT_DIR)

In [ ]:
# ============================================================
# Cell 3: Read data and calculate station-level means
# ============================================================

df = pd.read_excel(DATA_FILE)

required_columns = ["station_id", "time", "pm10", "lon", "lat"]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

df["time"] = pd.to_datetime(df["time"], errors="coerce")

# 
df = df.loc[
    (df["time"] >= START_DATE) &
    (df["time"] <= END_DATE)
].copy()

# 
df = df.dropna(subset=["station_id", "lon", "lat", "pm10"])

# 
station_mean = (
    df.groupby("station_id", as_index=False)
      .agg(
          lon=("lon", "first"),
          lat=("lat", "first"),
          pm10_mean=("pm10", "mean"),
          n_obs=("pm10", "size")
      )
)

print("Number of observations:", len(df))
print("Number of stations:", len(station_mean))

display(station_mean.head())
display(station_mean["pm10_mean"].describe())

In [ ]:
# ============================================================
# Cell 4: Spatial objects and projected coordinates
# ============================================================

germany = gpd.read_file(GERMANY_FILE)

if germany.crs is None:
    germany = germany.set_crs(epsg=4326)

germany = germany.to_crs(epsg=4326)

stations_gdf = gpd.GeoDataFrame(
    station_mean.copy(),
    geometry=gpd.points_from_xy(
        station_mean["lon"],
        station_mean["lat"]
    ),
    crs="EPSG:4326"
)

# نگه داشتن ایستگاه‌های داخل یا روی مرز آلمان
germany_union = germany.geometry.union_all()

inside_mask = (
    stations_gdf.geometry.within(germany_union) |
    stations_gdf.geometry.touches(germany_union)
)

stations_gdf = stations_gdf.loc[inside_mask].copy().reset_index(drop=True)

# دستگاه تصویرشده مناسب برای اروپا
PROJECTED_CRS = "EPSG:3035"

germany_proj = germany.to_crs(PROJECTED_CRS)
stations_proj = stations_gdf.to_crs(PROJECTED_CRS)

# مختصات بر حسب کیلومتر
stations_proj["x_km"] = stations_proj.geometry.x / 1000.0
stations_proj["y_km"] = stations_proj.geometry.y / 1000.0

print("Stations used:", len(stations_proj))
print("Projected CRS:", stations_proj.crs)

display(
    stations_proj[
        ["station_id", "lon", "lat", "pm10_mean", "x_km", "y_km"]
    ].head()
)

In [ ]:
# ============================================================
# Cell 5: Spatial trend used in the paper
# ============================================================

BETA_POSTERIOR = np.array([
     2.49,   # beta_0
     2.06,   # beta_1
    -8.22,   # beta_2
     7.28,   # beta_3
     6.25    # beta_4
], dtype=float)


def fit_coordinate_transform(lon, lat):
    lon = np.asarray(lon, dtype=float)
    lat = np.asarray(lat, dtype=float)

    parameters = {
        "lon_min": lon.min(),
        "lon_max": lon.max(),
        "lat_min": lat.min(),
        "lat_max": lat.max()
    }

    lon_01 = (
        (lon - parameters["lon_min"]) /
        (parameters["lon_max"] - parameters["lon_min"])
    )

    lat_01 = (
        (lat - parameters["lat_min"]) /
        (parameters["lat_max"] - parameters["lat_min"])
    )

    raw_terms = np.column_stack([
        lon_01,
        lat_01,
        lon_01**2,
        lat_01**2
    ])

    parameters["term_mean"] = raw_terms.mean(axis=0)
    parameters["term_sd"] = raw_terms.std(axis=0, ddof=0)

    if np.any(parameters["term_sd"] <= 0):
        raise ValueError("At least one coordinate term has zero variance.")

    return parameters


def build_design_matrix(lon, lat, parameters):
  
    lon = np.asarray(lon, dtype=float)
    lat = np.asarray(lat, dtype=float)

    lon_01 = (
        (lon - parameters["lon_min"]) /
        (parameters["lon_max"] - parameters["lon_min"])
    )

    lat_01 = (
        (lat - parameters["lat_min"]) /
        (parameters["lat_max"] - parameters["lat_min"])
    )

    raw_terms = np.column_stack([
        lon_01,
        lat_01,
        lon_01**2,
        lat_01**2
    ])

    standardized_terms = (
        raw_terms - parameters["term_mean"]
    ) / parameters["term_sd"]

    return np.column_stack([
        np.ones(len(lon)),
        standardized_terms
    ])


coord_transform = fit_coordinate_transform(
    stations_gdf["lon"].to_numpy(),
    stations_gdf["lat"].to_numpy()
)

X_station = build_design_matrix(
    stations_gdf["lon"].to_numpy(),
    stations_gdf["lat"].to_numpy(),
    coord_transform
)

trend_station = X_station @ BETA_POSTERIOR

stations_proj["trend_mean"] = trend_station
stations_proj["residual"] = (
    stations_proj["pm10_mean"].to_numpy() -
    stations_proj["trend_mean"].to_numpy()
)

print("Posterior beta:", BETA_POSTERIOR)
print("\nResidual summary:")
display(stations_proj["residual"].describe())

In [ ]:
# ============================================================
# Cell 6: Empirical variogram of spatial residuals
# ============================================================

coords_station = stations_proj[["x_km", "y_km"]].to_numpy()
residuals = stations_proj["residual"].to_numpy()

# فاصله زوجی ایستگاه‌ها
pair_distance = pdist(coords_station, metric="euclidean")

# نیم‌واریانس زوجی
pair_semivariance = 0.5 * pdist(
    residuals.reshape(-1, 1),
    metric="sqeuclidean"
)

max_distance = MAX_DISTANCE_FRACTION * pair_distance.max()

lag_edges = np.linspace(
    0.0,
    max_distance,
    N_LAG_BINS + 1
)

lag_index = np.digitize(pair_distance, lag_edges) - 1

empirical_rows = []

for lag_id in range(N_LAG_BINS):
    mask = lag_index == lag_id

    if mask.sum() < 3:
        continue

    empirical_rows.append({
        "lag_id": lag_id + 1,
        "distance_km": pair_distance[mask].mean(),
        "semivariance": pair_semivariance[mask].mean(),
        "n_pairs": int(mask.sum())
    })

emp_variogram = pd.DataFrame(empirical_rows)

if len(emp_variogram) < 4:
    raise RuntimeError(
        "Too few nonempty variogram bins. "
        "Reduce N_LAG_BINS or increase MAX_DISTANCE_FRACTION."
    )

display(emp_variogram)

In [ ]:
# ============================================================
# Cell 7: Constrained Matérn variogram fitting
# ============================================================

def matern_correlation(distance, phi, nu):
    """
    Matérn correlation:
    rho(h) = 2^(1-nu)/Gamma(nu) * (h/phi)^nu * K_nu(h/phi)
    """
    distance = np.asarray(distance, dtype=float)

    if phi <= 0 or nu <= 0:
        return np.full_like(distance, np.nan)

    scaled_distance = distance / phi
    rho = np.empty_like(scaled_distance)

    zero_mask = scaled_distance < 1e-10
    rho[zero_mask] = 1.0

    positive_mask = ~zero_mask
    z = scaled_distance[positive_mask]

    rho[positive_mask] = (
        (2.0 ** (1.0 - nu))
        / gamma(nu)
        * (z ** nu)
        * kv(nu, z)
    )

    rho = np.nan_to_num(
        rho,
        nan=0.0,
        posinf=1.0,
        neginf=0.0
    )

    return np.clip(rho, 0.0, 1.0)


def matern_semivariogram(distance, nugget, partial_sill, phi, nu):
    """
    gamma(h) = nugget + partial_sill * [1 - rho_Matern(h)]
    """
    distance = np.asarray(distance, dtype=float)

    rho = matern_correlation(
        distance=distance,
        phi=phi,
        nu=nu
    )

    semivariogram = (
        nugget
        + partial_sill * (1.0 - rho)
    )

    semivariogram = np.where(
        distance <= 1e-10,
        0.0,
        semivariogram
    )

    return semivariogram


h_emp = emp_variogram["distance_km"].to_numpy()
gamma_emp = emp_variogram["semivariance"].to_numpy()
n_pairs = emp_variogram["n_pairs"].to_numpy()

# ------------------------------------------------------------
# تنظیم آستانه‌ها
# ------------------------------------------------------------

PHI_MAX_KM = 90.0

# حدود پارامتر هموارسازی
NU_MIN = 0.20
NU_MAX = 2.50

# سطح انتهایی واریوگرام تجربی
n_tail = min(3, len(gamma_emp))
empirical_tail_level = np.mean(gamma_emp[-n_tail:])

# سیل کل اجازه ندارد خیلی بیشتر از سطح انتهایی تجربی شود.
TOTAL_SILL_MAX_FACTOR = 1.25
TOTAL_SILL_MAX = (
    TOTAL_SILL_MAX_FACTOR * empirical_tail_level
)

# کران اثر قطعه‌ای بر اساس نخستین طبقه فاصله
NUGGET_MAX = min(
    1.5 * gamma_emp[0],
    0.30 * TOTAL_SILL_MAX
)

# ------------------------------------------------------------
# وزن‌دهی
# ------------------------------------------------------------

weights = np.sqrt(
    n_pairs / n_pairs.max()
)

# ------------------------------------------------------------
# مقادیر اولیه
# ------------------------------------------------------------

initial_nugget = np.clip(
    0.5 * gamma_emp[0],
    0.01,
    max(NUGGET_MAX * 0.8, 0.02)
)

initial_total_sill = empirical_tail_level

initial_partial_sill = max(
    initial_total_sill - initial_nugget,
    1.0
)

initial_phi = min(
    np.median(h_emp),
    0.75 * PHI_MAX_KM
)

initial_nu = 1.0

initial_values = np.array([
    initial_nugget,
    initial_partial_sill,
    initial_phi,
    initial_nu
])

# ------------------------------------------------------------
# حدود پارامترها
# ------------------------------------------------------------

lower_bounds = np.array([
    0.0,       # nugget
    0.01,      # partial sill
    10.0,      # phi_s, km
    NU_MIN     # nu_s
])

upper_bounds = np.array([
    NUGGET_MAX,
    TOTAL_SILL_MAX,
    PHI_MAX_KM,
    NU_MAX
])


def weighted_residuals(parameters):
    nugget, partial_sill, phi, nu = parameters

    fitted = matern_semivariogram(
        h_emp,
        nugget=nugget,
        partial_sill=partial_sill,
        phi=phi,
        nu=nu
    )

    # جریمه در صورت عبور سیل کل از آستانه
    total_sill = nugget + partial_sill

    sill_penalty = max(
        0.0,
        total_sill - TOTAL_SILL_MAX
    )

    residual_part = weights * (
        fitted - gamma_emp
    )

    return np.append(
        residual_part,
        5.0 * sill_penalty
    )


fit = least_squares(
    weighted_residuals,
    x0=initial_values,
    bounds=(lower_bounds, upper_bounds),
    max_nfev=100000,
    xtol=1e-12,
    ftol=1e-12,
    gtol=1e-12
)

if not fit.success:
    raise RuntimeError(
        f"Variogram fitting failed: {fit.message}"
    )

NUGGET_HAT, SILL_HAT, PHI_HAT, NU_HAT = fit.x

SIGMA_HAT = np.sqrt(SILL_HAT)
TOTAL_SILL_HAT = NUGGET_HAT + SILL_HAT

print("Constrained Matérn variogram estimates")
print("---------------------------------------")
print(f"Estimated nugget       = {NUGGET_HAT:.4f}")
print(f"Estimated partial sill = {SILL_HAT:.4f}")
print(f"Estimated sigma        = {SIGMA_HAT:.4f}")
print(f"Estimated total sill   = {TOTAL_SILL_HAT:.4f}")
print(f"Estimated phi_s        = {PHI_HAT:.4f} km")
print(f"Estimated nu_s         = {NU_HAT:.4f}")
print("---------------------------------------")
print(f"Maximum allowed phi    = {PHI_MAX_KM:.1f} km")
print(f"Maximum total sill     = {TOTAL_SILL_MAX:.4f}")
print(f"Optimizer success      = {fit.success}")

In [ ]:
# ============================================================
# Cell 8: Plot empirical and fitted Matérn variogram
# ============================================================

h_curve = np.linspace(
    0.0,
    max_distance,
    500
)

gamma_curve = matern_semivariogram(
    h_curve,
    nugget=NUGGET_HAT,
    partial_sill=SILL_HAT,
    phi=PHI_HAT,
    nu=NU_HAT
)

fig, ax = plt.subplots(figsize=(8.2, 5.5))

point_sizes = 25 + 100 * (
    emp_variogram["n_pairs"]
    / emp_variogram["n_pairs"].max()
)

ax.scatter(
    emp_variogram["distance_km"],
    emp_variogram["semivariance"],
    s=point_sizes,
    edgecolor="black",
    linewidth=0.5,
    label="Empirical variogram",
    zorder=3
)

ax.plot(
    h_curve,
    gamma_curve,
    linewidth=2.2,
    label="Fitted Matérn variogram",
    zorder=2
)

ax.axhline(
    TOTAL_SILL_HAT,
    linestyle="--",
    linewidth=1.2,
    label="Estimated total sill"
)

ax.set_xlabel("Spatial distance (km)", fontsize=11)
ax.set_ylabel("Semivariance", fontsize=11)
ax.set_title(
    "Empirical and fitted Matérn variogram",
    fontsize=12
)

ax.grid(alpha=0.25)
ax.legend(frameon=False)

plt.tight_layout()

variogram_path = (
    OUTPUT_DIR
    / "pm10_fitted_matern_variogram.png"
)

plt.savefig(
    variogram_path,
    dpi=600,
    bbox_inches="tight"
)

plt.show()

print("Saved:", variogram_path)

In [ ]:
# ============================================================
# Cell 9: Construct prediction grid inside Germany
# ============================================================

minx, miny, maxx, maxy = germany_proj.total_bounds

x_values = np.linspace(minx, maxx, GRID_NX)
y_values = np.linspace(miny, maxy, GRID_NY)

xx, yy = np.meshgrid(x_values, y_values)

grid_all = gpd.GeoDataFrame(
    {
        "x_m": xx.ravel(),
        "y_m": yy.ravel()
    },
    geometry=gpd.points_from_xy(
        xx.ravel(),
        yy.ravel()
    ),
    crs=PROJECTED_CRS
)

germany_union_proj = germany_proj.geometry.union_all()

inside_grid = (
    grid_all.geometry.within(germany_union_proj) |
    grid_all.geometry.touches(germany_union_proj)
)

grid_proj = grid_all.loc[inside_grid].copy().reset_index(drop=True)

# نسخه جغرافیایی برای ساخت روند با lon و lat
grid_wgs84 = grid_proj.to_crs(epsg=4326)

grid_proj["lon"] = grid_wgs84.geometry.x.to_numpy()
grid_proj["lat"] = grid_wgs84.geometry.y.to_numpy()

grid_proj["x_km"] = grid_proj.geometry.x / 1000.0
grid_proj["y_km"] = grid_proj.geometry.y / 1000.0

print("Total rectangular grid points:", len(grid_all))
print("Grid points inside Germany:", len(grid_proj))

In [ ]:
# ============================================================
# Cell 10: Model-based spatial kriging
# ============================================================

# ساخت روند روی شبکه
X_grid = build_design_matrix(
    grid_proj["lon"].to_numpy(),
    grid_proj["lat"].to_numpy(),
    coord_transform
)

trend_grid = X_grid @ BETA_POSTERIOR

station_coords = stations_proj[["x_km", "y_km"]].to_numpy()
grid_coords = grid_proj[["x_km", "y_km"]].to_numpy()

n_station = len(station_coords)

# ماتریس فاصله ایستگاه‌ها
D_station = cdist(
    station_coords,
    station_coords,
    metric="euclidean"
)

# کوواریانس فرایند فضایی
R_station = matern_correlation(
    D_station,
    phi=PHI_HAT,
    nu=NU_HAT
)

C_station = SILL_HAT * R_station
# اثر قطعه‌ای فقط روی قطر ماتریس مشاهدات
C_station = (
    C_station +
    NUGGET_HAT * np.eye(n_station) +
    1e-8 * np.eye(n_station)
)

# باقیمانده‌های مشاهده‌شده
residual_vector = stations_proj["residual"].to_numpy()

# تجزیه چولسکی پایدار
chol_factor = cho_factor(
    C_station,
    lower=True,
    check_finite=False
)

C_inv_residual = cho_solve(
    chol_factor,
    residual_vector,
    check_finite=False
)

prediction_residual = np.empty(len(grid_proj))
prediction_variance = np.empty(len(grid_proj))

# پردازش دسته‌ای برای کنترل حافظه
CHUNK_SIZE = 10000

for start in range(0, len(grid_proj), CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, len(grid_proj))

    grid_chunk = grid_coords[start:end]

    D_cross = cdist(
        grid_chunk,
        station_coords,
        metric="euclidean"
    )

    R_cross = matern_correlation(
        D_cross,
        phi=PHI_HAT,
        nu=NU_HAT
    )

    C_cross = SILL_HAT * R_cross

    # میانگین شرطی باقیمانده
    prediction_residual[start:end] = (
        C_cross @ C_inv_residual
    )

    # C^{-1} c برای هر نقطه شبکه
    C_inv_cross_t = cho_solve(
        chol_factor,
        C_cross.T,
        check_finite=False
    )

    covariance_reduction = np.sum(
        C_cross.T * C_inv_cross_t,
        axis=0
    )

    # واریانس پیش‌گویی فرایند پنهان
    var_process = SILL_HAT - covariance_reduction

    # اگر هدف پیش‌گویی یک مشاهده جدید باشد،
    # اثر قطعه‌ای نیز به واریانس اضافه می‌شود.
    var_observation = var_process + NUGGET_HAT

    prediction_variance[start:end] = np.maximum(
        var_observation,
        0.0
    )

    print(f"Processed {end:,} / {len(grid_proj):,} grid points")


grid_proj["trend"] = trend_grid
grid_proj["kriged_residual"] = prediction_residual

grid_proj["prediction_mean"] = (
    trend_grid + prediction_residual
)

grid_proj["prediction_sd"] =np.sqrt(
    prediction_variance
)

print("\nPrediction summary:")
display(
    grid_proj[
        ["prediction_mean", "prediction_sd"]
    ].describe()
)

In [ ]:
# ============================================================
# Cell 11: Basic diagnostics
# ============================================================

print("Observed station means:")
print(stations_proj["pm10_mean"].describe())

print("\nPredicted grid means:")
print(grid_proj["prediction_mean"].describe())

print("\nPrediction standard deviations:")
print(grid_proj["prediction_sd"].describe())

if not np.isfinite(
    grid_proj[["prediction_mean", "prediction_sd"]].to_numpy()
).all():
    raise RuntimeError("Non-finite prediction values were generated.")

if (grid_proj["prediction_sd"] < 0).any():
    raise RuntimeError("Negative prediction standard deviation found.")

In [ ]:
# ============================================================
# Cell 12: Combined prediction and uncertainty maps
# ============================================================

fig, axes = plt.subplots(
    1,
    2,
    figsize=(14.5, 7.5),
    constrained_layout=True
)

# ------------------------------------------------------------
# Panel A: prediction mean
# ------------------------------------------------------------
grid_proj.plot(
    column="prediction_mean",
    ax=axes[0],
    cmap="viridis",
    markersize=5,
    legend=True,
    legend_kwds={
        "label": r"Predicted mean PM$_{10}$ ($\mu g/m^3$)",
        "shrink": 0.78
    }
)

germany_proj.boundary.plot(
    ax=axes[0],
    linewidth=0.8
)

stations_proj.plot(
    ax=axes[0],
    facecolor="white",
    edgecolor="black",
    markersize=24,
    linewidth=0.7,
    zorder=5
)

axes[0].set_title(
    "(a) Predicted spatial mean",
    fontsize=12
)

axes[0].set_axis_off()

# ------------------------------------------------------------
# Panel B: prediction standard deviation
# ------------------------------------------------------------
grid_proj.plot(
    column="prediction_sd",
    ax=axes[1],
    cmap="magma",
    markersize=5,
    legend=True,
    legend_kwds={
        "label": r"Prediction SD ($\mu g/m^3$)",
        "shrink": 0.78
    }
)

germany_proj.boundary.plot(
    ax=axes[1],
    linewidth=0.8
)

stations_proj.plot(
    ax=axes[1],
    facecolor="white",
    edgecolor="black",
    markersize=24,
    linewidth=0.7,
    zorder=5
)

axes[1].set_title(
    "(b) Prediction uncertainty",
    fontsize=12
)

axes[1].set_axis_off()

combined_map_path = (
    OUTPUT_DIR /
    "pm10_prediction_mean_and_uncertainty.png"
)

plt.savefig(
    combined_map_path,
    dpi=100,
    bbox_inches="tight"
)

plt.show()

print("Saved:", combined_map_path)

In [ ]:
# ============================================================
# Cell 13: Save separate publication-quality maps
# ============================================================

# ---------------- Prediction mean ----------------
fig, ax = plt.subplots(figsize=(10.2, 8.2))

grid_proj.plot(
    column="prediction_mean",
    ax=ax,
    cmap="viridis",
    markersize=6,
    legend=True,
    legend_kwds={
        "label": r"Predicted mean PM$_{10}$ ($\mu g/m^3$)",
        "shrink": 0.75
    }
)

germany_proj.boundary.plot(
    ax=ax,
    linewidth=0.9
)

stations_proj.plot(
    ax=ax,
    facecolor="white",
    edgecolor="black",
    linewidth=0.7,
    markersize=26,
    zorder=5
)

ax.set_axis_off()
plt.tight_layout()

mean_map_path = OUTPUT_DIR / "pm10_prediction_mean.png"

plt.savefig(
    mean_map_path,
    dpi=100,
    bbox_inches="tight"
)

plt.show()


# ---------------- Prediction SD ----------------
fig, ax = plt.subplots(figsize=(10.2, 8.2))

grid_proj.plot(
    column="prediction_sd",
    ax=ax,
    cmap="magma",
    markersize=6,
    legend=True,
    legend_kwds={
        "label": r"Prediction SD ($\mu g/m^3$)",
        "shrink": 0.75
    }
)

germany_proj.boundary.plot(
    ax=ax,
    linewidth=0.9
)

stations_proj.plot(
    ax=ax,
    facecolor="white",
    edgecolor="black",
    linewidth=0.7,
    markersize=26,
    zorder=5
)

ax.set_axis_off()
plt.tight_layout()

sd_map_path = OUTPUT_DIR / "pm10_prediction_sd.png"

plt.savefig(
    sd_map_path,
    dpi=100,
    bbox_inches="tight"
)

plt.show()

print("Saved:", mean_map_path)
print("Saved:", sd_map_path)

In [ ]:
# ============================================================
# Cell 14: Save numerical outputs
# ============================================================

# پارامترهای واریوگرام
parameter_table = pd.DataFrame({
    "parameter": [
        "sigma_fixed",
        "partial_sill_fixed",
        "nugget_hat",
        "phi_s_hat_km",
        "nu_s_hat"
    ],
    "estimate": [
        SIGMA_FIXED,
        SILL_FIXED,
        NUGGET_HAT,
        PHI_HAT,
        NU_HAT
    ]
})

parameter_path = OUTPUT_DIR / "estimated_variogram_parameters.csv"
parameter_table.to_csv(parameter_path, index=False)

# واریوگرام تجربی
emp_variogram_path = OUTPUT_DIR / "empirical_variogram.csv"
emp_variogram.to_csv(emp_variogram_path, index=False)

# میانگین ایستگاه‌ها و باقیمانده‌ها
station_output_path = OUTPUT_DIR / "station_means_and_residuals.csv"

stations_proj.drop(columns="geometry").to_csv(
    station_output_path,
    index=False
)

# شبکه پیش‌گویی در GeoPackage
prediction_gpkg_path = OUTPUT_DIR / "pm10_prediction_grid.gpkg"

grid_proj[
    [
        "lon",
        "lat",
        "trend",
        "kriged_residual",
        "prediction_mean",
        "prediction_sd",
        "geometry"
    ]
].to_file(
    prediction_gpkg_path,
    layer="pm10_prediction",
    driver="GPKG"
)

print("All outputs saved successfully.")
print(parameter_path)
print(emp_variogram_path)
print(station_output_path)
print(prediction_gpkg_path)

display(parameter_table)